In [175]:
import json
import os
import textwrap
import pandas as pd
import annotate_scenario
import translate_to_vis
import utils
import importlib
import numpy as np
importlib.reload(annotate_scenario)
importlib.reload(translate_to_vis)



<module 'translate_to_vis' from '/Users/anna/Dropbox/2023_AOI/MoralLearning/CodeSets/graph_extract/translate_to_vis.py'>

In [173]:
#set main paths
CUR_DIR = os.path.dirname(os.path.abspath(__name__))
DATA_DIR = CUR_DIR+'/data/'
DATA_DIR_HUMAN = DATA_DIR+'/human_annotation/'

## function definitions 

In [15]:
def open_scenario():
    """
    Opens a scenario file and returns its content.
    """
    scenario_path = os.path.join(DATA_DIR, FILENAME)
    with open(scenario_path, 'r') as file:
        all_scenarios = json.load(file)
        
    # error handling for assumptions about json entries
    try:
        scenario_json = all_scenarios[SCENARIO_ID]
    except:
        # print("Check scenario filename or scenario id")
        raise IndexError('Check scenario id exists in json file!')
        scenario_json = {}
    
    #make sure action id is in the scenario
    if ACT_ID not in scenario_json['options']:
        raise ValueError(f"Action ID {ACT_ID} not found in scenario options.")


    # display the scnenario text read in 
    this_scenario_text = scenario_json["text"]    
    print('Scenario Text: \n\n')
    print(textwrap.fill(this_scenario_text, width = 100), '\n\n')


    return scenario_json

In [ ]:
#soure human data for this scenarios if it exists
def get_human_data_values():
    """
    Get human annotation data for a given scenario and action choice.

    """
    #load human annotation data
    this_human_filename = f"{DATA_DIR_HUMAN}scenarios_{SCENARIO_ID}_choice_{ACT_ID}_value_scores.csv"
    all_human_data = {}
    
    if os.path.exists(this_human_filename):
        #load csv file
        this_human_data = pd.read_csv(this_human_filename)
        #convert to dictionary
        # human_data_dict = this_human_data.to_dict(orient='list')
    else:
        print('No value score human annotation data found for this scenario and action choice.')
    
    return this_human_data

def get_human_data_outcomes():
    """
    Get human annotation outcomes for a given scenario and action choice.
    """
    #get human data values
    this_human_filename = f"{DATA_DIR_HUMAN}scenarios_{SCENARIO_ID}_choice_{ACT_ID}_outcomes.csv"

    if os.path.exists(this_human_filename):
        outcome_data = pd.read_csv(this_human_filename, usecols=lambda column: column != 'Unnamed: 0')
    else:
        print('No outcome likelihood human annotation data found for this scenario and action choice.')
    return outcome_data


def get_human_data_utilities():
    """
    Get human annotation utilities for a given scenario and action choice.
    """
    #get human data values
    this_human_filename = f"{DATA_DIR_HUMAN}scenarios_{SCENARIO_ID}_choice_{ACT_ID}_outcome_utilities.csv"
     
    if os.path.exists(this_human_filename):
        utility_data = pd.read_csv(this_human_filename, usecols=lambda column: column != 'Unnamed: 0')
    else:
        print('No utility human annotation data found for this scenario and action choice.')
    return utility_data

def get_human_data_links():
    """
    Get human annotation links for a given scenario and action choice.
    """
    #get human data values
    this_human_filename = f"{DATA_DIR_HUMAN}scenarios_{SCENARIO_ID}_choice_{ACT_ID}_outcome_links.csv"
    
    if os.path.exists(this_human_filename):
        link_data = pd.read_csv(this_human_filename, usecols=lambda column: column != 'Unnamed: 0')
    else:
        print('No outcome links human annotation data found for this scenario and action choice.')
    return link_data



In [153]:

def evaluate_values(processed_values, this_scenario, this_act_I, human_data):  
   
    
    hd_value_names = human_data['value_names']
    hd_value_scores = np.array([round(x,2) for x in human_data['mean']])
    hd_values_missing = human_data['values_missing'][0]
    #create a dictionary where hd_value names are keys and hd_value_scores are values
    human_data_dict = dict(zip(hd_value_names, hd_value_scores))

    #print the missing values
    print('Missing values:')
    print(hd_values_missing)


    #print the highly rated values
    if type(hd_values_missing)==str and not pd.isna(hd_values_missing):
      hd_values_missing = set([x.lower() for x in hd_values_missing.split(',')])
    else:
      hd_values_missing = set()

    hd_highrated_valuenames = list(hd_value_names[hd_value_scores > 70])
    print('Highly-rated values:')
    print(hd_highrated_valuenames)


    #does the list of values generated overlap with those rated highly by humans? by how much?
    model_values = set(processed_values[1])
    common_list = model_values.intersection(hd_highrated_valuenames)
    print('Common high-rated values:')
    print(common_list)

    #compute percent overlap - of the model generated values, how many were highly rated?
    overlap = len(common_list)/len(model_values)
    print('Percent overlap with high-rated values: %.2f' %overlap)

    # re-score values from annotation data already collected to evaluate the importance scoring
    annot_values_scored = annotate_scenario.prompts.score_values(this_scenario, this_act_I,', '.join(hd_value_names))

    #compare to scores from human data
    #ensure that the two dictionaries have the same keys  
    assert all(hd_value_names == list(annot_values_scored.keys()))
    annot_values_scored_values = list(annot_values_scored.values())
    print("annotator scored values:")
    print(annot_values_scored)
    print("human data:")
    print(human_data_dict)
    #calculate correlation
    corr = np.corrcoef(annot_values_scored_values,hd_value_scores)
    print('Value score correlation with human data: %.2f' %corr[0,1])


## Look at Scenario

In [ ]:
#set scenario filename
FILENAME = 'scenarios.json'

#select scenario and action choice
SCENARIO_ID = 0
ACT_ID = '1'

#read in the scenario
scenario_json = open_scenario()


Scenario Text: 


My older sister (23) has a dog and forces almost every responsibility on my mother and younger
sister(13). I refuse to do anything to help with the dog as me and my older sister got in a big
argument before my sister got it as i knew my sister wasn't ready to take care of a dog. But
sometimes I do cave in and help out as I feel bad for the dog. The dog is untrained as after many
years and daily will pee on the floors and poops inside the home, we live in a small apartment and
its really awful. My older sister these past few months lives with her BF and only returns like a
day or two each week. Thus leaving everything to my sister and mother. I think its really unfair for
her to just abandon the dog, my mother is always complaining about it to get me or my little sister
to clean after it, everyday I have to dodge stepping in piss. Also, smelling piss and seeing shit on
the floor everyday is not fun. I really dislike having to see this dog being left like this, nobody
r

#### There are 4 major stages of processing.

0. Entities
Label the entities (no human data)

1. Value Scores / "Deontology"
Label the action with its virtues 

2. Outcomes
Map action to probable outcomes 

3. Outcome Utilities
Consequentialist analysis of harms/benefits of each outcome to each entity

4. Outcome Links
Connection between each entity and each outcome, in terms of Cause, Intend, and Desire



## Go through annotation process step by step (replicates main function in annotate_scenario) 

In [176]:

# get the action choice and convert to two pronoun options (I and Ziv)
this_act = scenario_json['options'][ACT_ID]
this_act_I = "I decide to " + this_act
this_act_Ziv = annotate_scenario.prompts.convert_I_Ziv(this_act_I)
print('\n\nAction choice:') 
print(this_act_Ziv)
print(this_act_I)

#get the scenario and convert to two pronoun options
this_scenario = scenario_json['text']
this_scenario_Ziv = annotate_scenario.prompts.convert_I_Ziv(this_scenario)


# create a dictionary to write out to csv later
scenario_dict = {'scenario': this_scenario, 'scenario_idx': scenario_json['id'],
                    'choice': this_act_I}





Action choice:
Ziv decides to refuse to take care of the dog unless their sister steps up and takes responsibility.
I decide to refuse to take care of the dog unless my sister steps up and takes responsibility.


In [177]:

#initialize Graph object    
g = annotate_scenario.node.Graph()
g.reset()   
print('Graph g initialized and reset.')

Graph g initialized and reset.


In [178]:
#Step 0. Get entities

# identify all sentient beings, returning both pronoun forms and a string list
returned_beings = annotate_scenario.process_beings(this_scenario,this_act,g)
beings_I = returned_beings[0]
beings_Ziv = returned_beings[1]
beings_str_list = returned_beings[2]

#update the scenario dict with the beings
scenario_dict["entities"] = beings_str_list



Identified these entities: 

I
older sister
mother
younger sister
dog
older sister's boyfriend


In [ ]:
#optionally, take a look at the generated graph
# g.print_graph()


#reload the prompts module to ensure the latest version is used -- run this after making changes to the prompts module or annotate_scenario.py
importlib.reload(annotate_scenario.prompts)
importlib.reload(annotate_scenario)

<module 'annotate_scenario' from '/Users/anna/Dropbox/2023_AOI/MoralLearning/CodeSets/graph_extract/annotate_scenario.py'>

In [ ]:
#Step 1.  #ACTION VALUE SCORES

#call the process_values function to identify the values associated with this action
processed_values  = annotate_scenario.process_values(this_scenario, this_act_I, this_act,g) 

all_values_scored = processed_values[0]
scenario_dict["values"]= processed_values[1]
print(all_values_scored)

    


values:
{'values': ['Responsibility', 'Compassion', 'Integrity', 'Accountability']}
{'anti-values': ['neglect', 'irresponsibility', 'indifference']}
{'responsibility': 40, 'compassion': 60, 'integrity': 70, 'accountability': 50, 'neglect': 20, 'irresponsibility': 30, 'indifference': 25}


In [ ]:
#compare this to human data collected
data_values = get_human_data_values()
evaluate_values(processed_values,this_scenario, this_act_I, data_values)

Missing values:
refusing to do anything with the dog even though isnt her dog ,Stubbornness ,A sense of justice maybe,compromise
Highly-rated values:
['accountability', 'boundaries']
Common high-rated values:
{'accountability'}
Percent overlap with high-rated values: 0.14
annotator scored values:
{'responsibility': 60, 'fairness': 70, 'autonomy': 80, 'accountability': 75, 'boundaries': 90, 'cooperation': 30, 'family harmony': 20, 'irresponsibility': 10, 'lack of compassion': 40, 'selfishness': 25, 'neglect': 5}
human data:
{'responsibility': 51.86, 'fairness': 46.43, 'autonomy': 66.43, 'accountability': 75.0, 'boundaries': 70.71, 'cooperation': 26.43, 'family harmony': 20.14, 'irresponsibility': 50.14, 'lack of compassion': 54.14, 'selfishness': 49.14, 'neglect': 62.0}
Value score correlation with human data: 0.54


In [154]:
#Step 2. Outcomes

processed_events = annotate_scenario.process_outcomes(this_scenario, this_act)
events_I= processed_events[1]
events_Ziv= processed_events[0]
print("\n".join(events_I))         
scenario_dict["outcomes"]= events_I

My refusal to help may increase the burden on my mother and younger sister.
My decision may lead to further tension or conflict within the family.
The dog's well-being may continue to be neglected if my sister does not take responsibility.
My relationship with my older sister may become more strained.
The living conditions in the apartment may deteriorate further without my occasional assistance.
My mother and younger sister may feel more stressed and overwhelmed with the responsibility.
I may experience guilt or distress from seeing the dog's condition worsen.


In [161]:
#Step 3. Outcome utilities
impacts_list = annotate_scenario.process_impacts(this_scenario_Ziv, this_act, this_act_Ziv, events_Ziv, events_I,beings_Ziv,g) 



Processing impacts of event: My refusal to help may increase the burden on my mother and younger sister.
Scored impacts for these beings:
['Ziv', 'older sister', 'mother', 'younger sister', 'dog']
[0, 0, -8, -8, -5]

Processing impacts of event: My decision may lead to further tension or conflict within the family.
Scored impacts for these beings:
['Ziv', 'older sister', 'mother', 'younger sister', 'dog']
[-5, -3, -4, -4, -2]

Processing impacts of event: The dog's well-being may continue to be neglected if my sister does not take responsibility.
Scored impacts for these beings:
['Ziv', 'older sister', 'mother', 'younger sister', 'dog']
[-2, 0, -3, -3, -10]

Processing impacts of event: My relationship with my older sister may become more strained.
Scored impacts for these beings:
['Ziv', 'older sister', 'mother', 'younger sister', 'dog']
[-5, -4, 0, 0, 0]

Processing impacts of event: The living conditions in the apartment may deteriorate further without my occasional assistance.
Sco

In [1]:
#Step 4. causal / intentional / knowledge links 
annotate_scenario.process_causal_links(this_scenario_Ziv, events_Ziv, events_I, this_act_Ziv,g)    

#TODO: D should be relabed as I for intend. Currently, C = Cause, K = expect (or know), and D = intend.

NameError: name 'annotate_scenario' is not defined

In [ ]:
#optional -- write out the results 

this_output_filename = f"scenarios_{SCENARIO_ID}_choice_{ACT_ID}.json"
print('\n\nWriting to file: '+this_output_filename)
g_print = g.print_graph()
utils.write_jsonlines(DATA_DIR+this_output_filename,g_print)
print('\n\n')


translate_to_vis.main(DATA_DIR+this_output_filename)



Writing to file: scenarios_0_choice_1_test.json



